In [ ]:
%pip install --upgrade pip

In [ ]:
%pip install --default-timeout=200 msprime tskit numpy pandas matplotlib networkx scikit-learn scipy kmapper ripser persim plotly ipywidgets

In [ ]:
import msprime
import tskit
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import networkx as nx
from collections import defaultdict
from sklearn.metrics import pairwise_distances
from sklearn.manifold import MDS
from sklearn.decomposition import PCA
import kmapper as km
from kmapper.adapter import to_networkx
from kmapper.plotlyviz import plotlyviz
from ripser import Rips
from persim import plot_diagrams
import warnings
warnings.filterwarnings("ignore")

In [ ]:
from funcs.simulation import simulate_one_pulse, simulate_two_pulses

from funcs.tract_processing import get_tracts_dict

from funcs.tract_extraction import get_population_tracts_dataframe

from funcs.bio_summary import build_individual_biology

from funcs.features import build_binary_matrix

from funcs.filters import compute_filter

from funcs.mapper_analysis import mapper_summary

from funcs.persistent_homology import persistence_summary

from parameters import params, t_second_wave, t, Ne


In [ ]:
w_size=1000
M=params['chrom_length']//w_size

Фильтры для Mapper

In [ ]:
filter_functions=pd.DataFrame({
    'filter_id':[1,2,3,4,5,6],
    'filter_name':[
        'MDS1',
        'distance_to_medoid',
        'mean_distance',
        'introgressed_fraction',
        'tract_count',
        'mean_tract_length'
    ],
    'filter_dim':[1,1,1,1,1,1]
})

Симуляция двух сценариев: один / две независимые волны интрорегрессии


In [ ]:
one_ts=simulate_one_pulse(42, 0.1)
two_ts=simulate_two_pulses(42,prop_first=0.05,prop_second=0.05)

Извлечение introgressed tracts

In [ ]:
one_tracts=get_tracts_dict(one_ts, params['n_eu'], [t['t_nd_migration']])
two_tracts=get_tracts_dict(two_ts,params['n_eu'],[t['t_nd_migration'],t_second_wave])

Построение бинарных матриц признаков: (строки = individuals, столбцы = genomic windows)

In [ ]:
Xa=build_binary_matrix(one_tracts, params['n_eu'], w_size, M)
Xb=build_binary_matrix(two_tracts, params['n_eu'], w_size, M) 

Cоздание "desert" сценария для one-pulse, зануляем участок генома

In [ ]:
Xd=Xa.copy()

st=15000000//w_size
ed=20000000//w_size

Xd[:,st:ed]=0

In [ ]:
scenarios={'one_pulse':Xa,
           'two_pulses':Xb,
           'one_pulse_desert':Xd}

tract_dicts={'one_pulse':one_tracts,
             'two_pulses':two_tracts,
             'one_pulse_desert':one_tracts}

Для каждого сценария строим табличку статистик: introregressed_windows, introregressed_fraction, tract_count, mean_tract_length

In [ ]:
individual_biology=[]

for scenario in scenarios:
    bio=build_individual_biology(tract_dicts[scenario],
                                 scenarios[scenario],scenario)

    individual_biology.append(bio)


individual_biology=pd.concat(individual_biology,ignore_index=True)

Рзелуьтаты

In [ ]:
mapper_summary_rows=[]
mapper_node_rows=[]
ph_rows=[]
analysis_rows=[]

### Подбор eps для DBSCAN

Для каждого сценария строится distance matrix между индивидами с помощью Jaccard distance. Из распределения этих расстояний выбираем значение eps для DBSCAN.

In [ ]:
eps_vals={}

for scenario,X in scenarios.items():
    dist=pairwise_distances(X.astype(bool), metric='jaccard')
    tri=np.triu_indices_from(dist,k=1)
    vals=dist[tri]
    vals=vals[vals>0]

    if len(vals)==0:
        eps=0.05
    else:
        eps=np.percentile(vals, 75)

    eps=max(float(eps), 0.01)
    eps_vals[scenario]=eps

### Основной анализ: Persistent Homology + Mapper

Для каждого сценария строим distance matrix. Дальше запускаем 2 независимых анализа:

1.  Persistent Homology

Из distance matrix считаем persistent homology. Получаем топологические характеристики данных:

   -- компоненты связности (H0),

   -- циклы/петли (H1), 

   -- их persistence

Результаты кладем в ph_rows.

2. Mapper analysis

Затем для каждого filter function строится lens (MDS1, tract_count и т.д.). Используя этот filter, distance matrix и DBSCAN, строится Mapper graph. После построения графа считаются:

   -- общие свойства графа (mapper_summary),

   -- биологические характеристики каждого узла (mapper_node_rows),

   -- параметры запуска анализа (analysis_rows)

In [ ]:
for scenario,X in scenarios.items(): 
    dist=pairwise_distances(X.astype(bool),metric='jaccard')

    ph=persistence_summary(dist)
    ph['scenario']=scenario
    ph_rows.append(ph)
    bio=individual_biology[individual_biology['scenario']==scenario]

    for _,frow in filter_functions.iterrows():
        filter_id=frow['filter_id']
        filter_name=frow['filter_name']

        lens=compute_filter(filter_name,X,dist,bio)
        mp=km.KeplerMapper(verbose=0)

        graph=mp.map(lens, X,
            clusterer=km.cluster.DBSCAN(
                eps=eps_vals[scenario],
                min_samples=2,
                metric='jaccard'
            ),
            cover=km.Cover(n_cubes=8, perc_overlap=0.3)
        )

        ms=mapper_summary(graph)
        mapper_summary_rows.append({'scenario':scenario, 'filter_id':filter_id,
                                    'eps':eps_vals[scenario], 'cover_intervals':8, 'cover_overlap':0.3, **ms })

        for node_id,members in graph['nodes'].items():
            inds=np.array(members)
            sub=bio.iloc[inds]
            mapper_node_rows.append({
                'scenario':scenario,
                'filter_id':filter_id,
                'node_id':node_id,
                'node_size':len(inds),
                'mean_introgressed_windows': sub['introgressed_windows'].mean(),
                'mean_introgressed_fraction': sub['introgressed_fraction'].mean(),
                'mean_tract_count': sub['tract_count'].mean(),
                'mean_tract_length': sub['mean_tract_length'].mean()
            })

        analysis_rows.append({
            'run_id': f"{scenario}_{filter_name}",
            'scenario':scenario,
            'feature_type':'binary',
            'metric':'jaccard',
            'window_size':w_size,
            'filter_id':filter_id,
            'eps':eps_vals[scenario],
            'cover_intervals':8,
            'cover_overlap':0.3,
            'mds_mode':'1D'
        })


### Сохраняем результаты

In [ ]:
import os
dir='results'
os.makedirs(dir,exist_ok=True)

In [ ]:
filter_functions.to_csv(f'{dir}/filter_functions.csv',index=False) 
individual_biology.to_csv(f'{dir}/individual_biology.csv',index=False)

mapper_summary_df=pd.DataFrame(
    mapper_summary_rows
)

mapper_node_biology=pd.DataFrame(
    mapper_node_rows
)

ph_summary=pd.DataFrame(
    ph_rows
)

analysis_parameters=pd.DataFrame(
    analysis_rows
)

mapper_summary_df.to_csv(f'{dir}/mapper_summary.csv',index=False)
mapper_node_biology.to_csv(f'{dir}/mapper_node_biology.csv',index=False) 
ph_summary.to_csv(f'{dir}/ph_summary.csv',index=False)
analysis_parameters.to_csv(f'{dir}/analysis_parameters.csv',index=False)

In [ ]:
import joblib
persistence_results = {}
for scenario, X in scenarios.items():
    dist = pairwise_distances(X.astype(bool), metric='jaccard')
    rips = Rips(maxdim=1)
    diagrams = rips.fit_transform(dist, distance_matrix=True)
    persistence_results[scenario] = {
        'distance_matrix': dist,
        'diagrams': diagrams
    }
    plt.figure()
    plot_diagrams(diagrams)
    plt.title(f"Persistence: {scenario}")
    plt.savefig(f'results/persistence/persistence_{scenario}.png', dpi=150, bbox_inches='tight')
    plt.close()

joblib.dump(persistence_results, 'results/persistence/persistence_results.joblib')

mapper_results = {}
for scenario, X in scenarios.items():
    dist = pairwise_distances(X.astype(bool), metric='jaccard')
    bio = individual_biology[individual_biology['scenario'] == scenario]
    
    lens = compute_filter('MDS1', X, dist, bio)
    mp = km.KeplerMapper(verbose=0)
    
    graph = mp.map(lens, X,
        clusterer=km.cluster.DBSCAN(
            eps=eps_vals[scenario],
            min_samples=2,
            metric='jaccard'
        ),
        cover=km.Cover(n_cubes=8, perc_overlap=0.3)
    )
    
    mapper_results[scenario] = {
        'graph': graph,
        'lens': lens,
        'eps': eps_vals[scenario]
    }
    
    fig = plotlyviz(graph, title=f"{scenario} Mapper")
    fig.write_html(f'results/mapper/mapper_{scenario}.html')

joblib.dump(mapper_results, 'results/mapper/mapper_results.joblib')
joblib.dump(scenarios, 'results/scenarios.joblib')